# Dirac-3S Planted Benchmarking


In [ ]:
import qci_client as qc
import numpy as np
import time
import os
import matplotlib.pyplot as plt
client = qc.QciClient(api_token="<API_TOKEN>", url="https://api.qci-prod.com")


## Set up Dirac-3 solver

In [2]:
def convert_hamiltonian_to_poly_format(linear_terms, quadratic_terms, tol=0.0):
    """
    Convert (c, Q) of H(x) = x^T Q x + c^T x into Dirac-3 polynomial format.
    """
    c = np.asarray(linear_terms, dtype=float).ravel()
    Q = np.asarray(quadratic_terms, dtype=float)
    n = c.shape[0]

    if Q.shape != (n, n):
        raise ValueError(f"quadratic_terms has shape {Q.shape}, expected ({n}, {n})")

    poly_indices = []
    poly_coefficients = []

    # Linear terms
    for i in range(n):
        if abs(c[i]) > tol:
            poly_indices.append([0, i + 1])
            poly_coefficients.append(float(c[i]))

    # Quadratic terms -- fold the lower triangle onto the upper triangle
    Q_upper = np.triu(Q) + np.tril(Q, -1).T
    for i in range(n):
        for j in range(i, n):
            val = Q_upper[i, j]
            if abs(val) > tol:
                poly_indices.append([i + 1, j + 1])
                poly_coefficients.append(float(val))

    return poly_indices, poly_coefficients

In [3]:
def solve_dirac3(num_var, indices, coefficients, relaxation_schedule=1, sum_constraint=100, num_samples=1, device='dirac-3S'):
    data = [{"idx":idx, "val":val} for idx, val in zip(indices, coefficients)]
    file_quadratic = {
        "file_name": "dirac_3_continuous_variable_example",
        "file_config": {
            "polynomial": {
                "num_variables": num_var,
                "min_degree": 1,
                "max_degree": 2,
                "data": data,
            }
        }
    }
    # upload to qci client
    file_response = client.upload_file(file=file_quadratic)
    job_body = client.build_job_body(
        job_type = 'sample-hamiltonian',
        job_name = 'planted_quadratic', # user-defined (optional)
        job_params = {
            'device_type': device,
            'relaxation_schedule':relaxation_schedule,
            'sum_constraint':sum_constraint,
            'num_samples':num_samples,
        },
        polynomial_file_id=file_response['file_id'],
    )

    job_response = client.process_job(job_body=job_body)
    get_job_metrics_response = client.get_job_metrics(job_id=job_response["job_info"]["job_id"])
    return job_response, get_job_metrics_response


### Load data file

In [4]:
def loadQ(path, n):
    tri = np.load(path, mmap_mode="r")
    Q = np.empty((n,n), dtype=tri.dtype)
    iu = np.triu_indices(n)
    Q[iu] = tri
    Q.T[iu] = tri
    return Q

### Solve with Dirac-3S

In [5]:
# Instance details
optimal_energy = 12000
sum_constraint = 100
num_var = 2000
k = 44
ub =10
seed =100

name = f"STQP_n_{num_var}_k_{k}_R_{sum_constraint}_seed_{seed}_ub_{ub}"
instance_path = os.path.join(f"Instances/{name}.npy")
c = np.zeros(num_var)
try:
    Q= loadQ(instance_path,num_var)
    print("loaded file sucessfully.")
except FileNotFoundError:
    print(f"File {instance_path} does not exist.")


loaded file sucessfully.


In [6]:
# solver parameters
relaxation_schedule = 1
num_samples = 10
device = 'dirac-3s'

In [7]:
# solve using Dirac-3S
poly_indices, poly_coefficients = convert_hamiltonian_to_poly_format(
                linear_terms=c,
                quadratic_terms=Q,
            )

response, metrics = solve_dirac3(
                        num_var=num_var,
                        indices=poly_indices,
                        coefficients=poly_coefficients,
                        relaxation_schedule=relaxation_schedule,
                        sum_constraint=sum_constraint,
                        device=device,
                        num_samples=num_samples)
print(f"Dirac-3 Run complete. Response received.")
metrics = metrics["job_metrics"]
time_ns = metrics["time_ns"]
device = time_ns["device"][f"{device}"] #f"{device}"
runtimes = device["samples"]["runtime"]
pre = device["samples"].get("preprocessing_time", 0)
post = device["samples"]["postprocessing_time"]
solutions = response["results"]["solutions"]
energies = response['results']['energies']

runtimes_sec = [t*1e-9 for t in runtimes]
post_sec  = [t*1e-9 for t in post]

2026-10-08 13:47:05 - Dirac allocation balance = 0 s (unmetered)
2026-10-08 13:47:06 - Job submitted: job_id='6ac7d71a358b23d0cb371f14'
2026-10-08 13:47:06 - QUEUED
2026-10-08 13:47:08 - RUNNING
2026-10-08 13:51:07 - COMPLETED
2026-10-08 13:51:09 - Dirac allocation balance = 0 s (unmetered)
Dirac-3 Run complete. Response received.


In [8]:
print(f"Dirac-3S all samples energies:{[round(e, 2) for e in energies]}")
print(f"Dirac-3S all samples runtimes:{[round(t, 2) for t in runtimes]}")

Dirac-3S all samples energies:[12000, 12000.03, 12628.88, 12938.26, 13029.78, 13034.36, 13070.21, 13075.66, 13151.44, 13157.75]
Dirac-3S all samples runtimes:[22022850037, 13852680206, 17948863983, 13679844856, 18551261902, 20598064423, 12255845070, 15364009857, 23321243286, 26423971176]


In [9]:
best = min(zip(energies, solutions, runtimes_sec, post_sec), key=lambda x: x[0])
dirac_energy, best_solution, best_runtime, best_postprocessing_time = best
preprocessing_time = pre*1e-9
arr = np.array(best_solution)
indices = np.where(arr> 1e-6)[0]

In [10]:
print(f"samples:{num_samples}, Relaxation schedule:{relaxation_schedule}")
print(f"best energy over {num_samples} samples:{dirac_energy:.6f}")
print(f"support size:{len(indices)}")
print(f"Preprocessing Time:{preprocessing_time}")
print(f"Time taken for this run:{best_runtime}")
print(f"Postprocessing Time for this run:{best_postprocessing_time}")

samples:10, Relaxation schedule:1
best energy over 10 samples:12000.000000
support size:44
Preprocessing Time:1.8553726670000001
Time taken for this run:22.022850037
Postprocessing Time for this run:6.5842375760000005


In [11]:
# compute relative gap
tol = 1e-7
abs_gap = abs(dirac_energy-optimal_energy)
relative_gap = (round(abs_gap,6)*100)/round(optimal_energy,6)

In [12]:
# print result
if abs_gap>tol:
    print(f"Dirac-3S solution is not optimal")
    print(f"Absolute Gap:{abs_gap}")
    print(f"Relative Gap(%):{abs(relative_gap)}")
else:
    print(f"Optimal solution found:{dirac_energy}.")

Optimal solution found:12000.
